# MedVQA — Exploratory Data Analysis
## VQA-RAD Dataset Analysis

This notebook performs exploratory data analysis on the VQA-RAD medical VQA dataset.
We analyze question types, answer distributions, image modalities, and class imbalances.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded successfully')

In [ ]:
# Paths
DATA_DIR = Path('../data/raw/vqa_rad')
ANNOTATION_FILE = DATA_DIR / 'VQA_RAD_Dataset.json'
IMAGE_DIR = DATA_DIR / 'images'

# Load annotations
with open(ANNOTATION_FILE, 'r') as f:
    annotations = json.load(f)

print(f'Annotation file loaded: {len(annotations)} entries')
print(f'Image directory exists: {IMAGE_DIR.exists()}')
print(f'Number of images: {len(list(IMAGE_DIR.glob("*.*")))}')

In [ ]:
# Parse annotations into a DataFrame-like structure
samples = []
if isinstance(annotations, list):
    entries = annotations
elif isinstance(annotations, dict):
    entries = annotations.get('data', annotations.get('questions', []))

for entry in entries:
    samples.append({
        'question': entry.get('question', ''),
        'answer': entry.get('answer', ''),
        'type': entry.get('type', entry.get('question_type', 'open')),
        'split': entry.get('split', entry.get('phase', 'unknown')),
        'image_name': entry.get('filename', entry.get('image_name', '')),
    })

print(f'Parsed {len(samples)} samples')
print(f'\nSample entry: {samples[0]}')

In [ ]:
# 1. Question Type Distribution
qtypes = [s['type'] for s in samples]
qtype_counts = Counter(qtypes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
ax1.bar(qtype_counts.keys(), qtype_counts.values(), color=['#2ecc71', '#3498db'])
ax1.set_title('Question Type Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Question Type')
ax1.set_ylabel('Count')
for i, (k, v) in enumerate(qtype_counts.items()):
    ax1.text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
ax2.pie(qtype_counts.values(), labels=qtype_counts.keys(), autopct='%1.1f%%', colors=colors[:len(qtype_counts)])
ax2.set_title('Question Type Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../experiments/eda_question_types.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Yes/No questions: {qtype_counts.get("yes/no", 0)}')
print(f'Open-ended questions: {qtype_counts.get("open", 0)}')

In [ ]:
# 2. Answer Vocabulary and Frequency Distribution
answers = [s['answer'].lower().strip() for s in samples]
answer_counts = Counter(answers)
top_answers = answer_counts.most_common(30)

fig, ax = plt.subplots(figsize=(14, 8))
words, counts = zip(*top_answers)
bars = ax.barh(range(len(words)), counts, color='steelblue')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words, fontsize=10)
ax.set_xlabel('Frequency')
ax.set_title('Top 30 Most Frequent Answers', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(count),
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../experiments/eda_answer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Unique answers: {len(answer_counts)}')
print(f'Answer vocabulary size: {len(answer_counts)}')

In [ ]:
# 3. Split Distribution
splits = [s['split'] for s in samples]
split_counts = Counter(splits)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(split_counts.keys(), split_counts.values(), color=['#3498db', '#2ecc71', '#e74c3c'])
ax.set_title('Dataset Split Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Split')
ax.set_ylabel('Count')
for i, (k, v) in enumerate(split_counts.items()):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../experiments/eda_splits.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Split distribution: {dict(split_counts)}')

In [ ]:
# 4. Image Resolution Statistics
image_paths = list(IMAGE_DIR.glob('*.*'))
resolutions = []
aspect_ratios = []

for img_path in tqdm(image_paths[:100], desc='Analyzing images'):
    try:
        img = Image.open(img_path)
        w, h = img.size
        resolutions.append((w, h))
        aspect_ratios.append(w / h if h > 0 else 1.0)
    except Exception as e:
        print(f'Error loading {img_path}: {e}')

widths, heights = zip(*resolutions) if resolutions else ([], [])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Width distribution
axes[0].hist(widths, bins=20, color='#3498db', edgecolor='white')
axes[0].set_title('Image Width Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')

# Height distribution
axes[1].hist(heights, bins=20, color='#2ecc71', edgecolor='white')
axes[1].set_title('Image Height Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')

# Aspect ratio distribution
axes[2].hist(aspect_ratios, bins=20, color='#e74c3c', edgecolor='white')
axes[2].set_title('Aspect Ratio Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Aspect Ratio (w/h)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../experiments/eda_image_resolutions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Width range: {min(widths)}-{max(widths)}')
print(f'Height range: {min(heights)}-{max(heights)}')
print(f'Mean aspect ratio: {np.mean(aspect_ratios):.2f}')

In [ ]:
# 5. Sample Visualization Grid
fig, axes = plt.subplots(5, 5, figsize=(20, 20))

for idx, ax in enumerate(axes.flat):
    if idx < len(samples):
        sample = samples[idx]
        img_path = IMAGE_DIR / sample['image_name']
        
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        
        question = sample['question'][:60] + '...' if len(sample['question']) > 60 else sample['question']
        ax.set_title(f'Q: {question}\nA: {sample["answer"]}', fontsize=8)
        ax.axis('off')

plt.suptitle('VQA-RAD Sample Grid (Image-Question-Answer Triplets)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../experiments/eda_sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample grid saved.')

In [ ]:
# Summary statistics
print('=' * 50)
print('VQA-RAD DATASET SUMMARY')
print('=' * 50)
print(f'Total samples: {len(samples)}')
print(f'Yes/No questions: {sum(1 for s in samples if s["type"].lower() in ["yes/no", "binary"])}')
print(f'Open-ended questions: {sum(1 for s in samples if s["type"].lower() == "open")}')
print(f'Unique answers: {len(set(answers))}')
print(f'Average question length: {np.mean([len(s["question"].split()) for s in samples]):.1f} words')
print(f'Average answer length: {np.mean([len(s["answer"].split()) for s in samples]):.1f} words')
print(f'Number of images: {len(image_paths)}')
print('=' * 50)